In [1]:
import pandas as pd

# Load your existing SARIMA validation (Mar-Jun 2026)
total_val = pd.read_csv("total_actual_vs_forecast.csv")
print(total_val)

# Load full crime data to get naive seasonal baseline (same month, prior year)
crime_full = pd.read_csv("data/cleaned_crime_long_through_june2026.csv")
crime_full["Date"] = pd.to_datetime(crime_full["Date"])
london_monthly_full = crime_full.groupby("Date")["CrimeCount"].sum().reset_index()

# Pull 2025 equivalents for March-June
naive_months = ["2025-03-01", "2025-04-01", "2025-05-01", "2025-06-01"]
naive_values = london_monthly_full[london_monthly_full["Date"].isin(naive_months)].sort_values("Date")["CrimeCount"].values

total_val["Naive_Seasonal"] = naive_values
print(total_val)

     Month  Actual  Forecast  Pct_Error
0  2026-03   75137     73103        2.7
1  2026-04   73990     71817        2.9
2  2026-05   78652     76096        3.2
3  2026-06   79894     75829        5.1
     Month  Actual  Forecast  Pct_Error  Naive_Seasonal
0  2026-03   75137     73103        2.7           75599
1  2026-04   73990     71817        2.9           74357
2  2026-05   78652     76096        3.2           78653
3  2026-06   79894     75829        5.1           78414


/tmp/ipykernel_898/2815423845.py:14: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  naive_values = london_monthly_full[london_monthly_full["Date"].isin(naive_months)].sort_values("Date")["CrimeCount"].values


In [2]:
# Apply the established ensemble weights: 0.4 SARIMA / 0.6 Naive Seasonal
total_val["Ensemble"] = (0.4 * total_val["Forecast"]) + (0.6 * total_val["Naive_Seasonal"])

# Calculate percentage error for each method
total_val["SARIMA_Error"] = (abs(total_val["Actual"] - total_val["Forecast"]) / total_val["Actual"] * 100).round(2)
total_val["Naive_Error"] = (abs(total_val["Actual"] - total_val["Naive_Seasonal"]) / total_val["Actual"] * 100).round(2)
total_val["Ensemble_Error"] = (abs(total_val["Actual"] - total_val["Ensemble"]) / total_val["Actual"] * 100).round(2)

print(total_val[["Month", "Actual", "Forecast", "Naive_Seasonal", "Ensemble", "SARIMA_Error", "Naive_Error", "Ensemble_Error"]])

print("\nMean errors across the window:")
print("SARIMA alone:  ", round(total_val["SARIMA_Error"].mean(), 2), "%")
print("Naive Seasonal:", round(total_val["Naive_Error"].mean(), 2), "%")
print("Ensemble:      ", round(total_val["Ensemble_Error"].mean(), 2), "%")

     Month  Actual  Forecast  Naive_Seasonal  Ensemble  SARIMA_Error  \
0  2026-03   75137     73103           75599   74600.6          2.71   
1  2026-04   73990     71817           74357   73341.0          2.94   
2  2026-05   78652     76096           78653   77630.2          3.25   
3  2026-06   79894     75829           78414   77380.0          5.09   

   Naive_Error  Ensemble_Error  
0         0.61            0.71  
1         0.50            0.88  
2         0.00            1.30  
3         1.85            3.15  

Mean errors across the window:
SARIMA alone:   3.5 %
Naive Seasonal: 0.74 %
Ensemble:       1.51 %


In [3]:
# July 2026 actual and existing SARIMA forecast
july_actual = 83665
july_sarima_forecast = 84666

# Pull July 2025 actual (naive seasonal forecast for July 2026)
july_2025_actual = london_monthly_full[london_monthly_full["Date"] == "2025-07-01"]["CrimeCount"].values[0]
print("July 2025 actual (naive seasonal forecast):", july_2025_actual)

# Ensemble blend
july_ensemble = (0.4 * july_sarima_forecast) + (0.6 * july_2025_actual)

# Errors
july_sarima_error = abs(july_actual - july_sarima_forecast) / july_actual * 100
july_naive_error = abs(july_actual - july_2025_actual) / july_actual * 100
july_ensemble_error = abs(july_actual - july_ensemble) / july_actual * 100

print(f"\nJuly 2026 comparison:")
print(f"SARIMA alone:   Forecast={july_sarima_forecast}, Error={july_sarima_error:.2f}%")
print(f"Naive Seasonal: Forecast={july_2025_actual}, Error={july_naive_error:.2f}%")
print(f"Ensemble:       Forecast={round(july_ensemble)}, Error={july_ensemble_error:.2f}%")

July 2025 actual (naive seasonal forecast): 82509

July 2026 comparison:
SARIMA alone:   Forecast=84666, Error=1.20%
Naive Seasonal: Forecast=82509, Error=1.38%
Ensemble:       Forecast=83372, Error=0.35%


In [4]:
july_ensemble_results = pd.DataFrame({
    "Month": ["2026-07"],
    "Actual": [july_actual],
    "SARIMA_Forecast": [july_sarima_forecast],
    "Naive_Forecast": [july_2025_actual],
    "Ensemble_Forecast": [round(july_ensemble)],
    "SARIMA_Error": [round(july_sarima_error, 2)],
    "Naive_Error": [round(july_naive_error, 2)],
    "Ensemble_Error": [round(july_ensemble_error, 2)]
})
july_ensemble_results.to_csv("data/july_ensemble_comparison.csv", index=False)
print("Saved!")

Saved!
